# 8) Results
- results and figures

# 1. Import packages

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')
import glob
import os, sys, gc
from pathlib import Path
from scipy.interpolate import PchipInterpolator
from matplotlib.patches import Patch
import calendar
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase
from matplotlib.ticker import FormatStrFormatter

In [ ]:
## making sure to stay under the project folder
root = Path.cwd().resolve().parents[0] 
sys.path.insert(0, str(root))
os.chdir(root)

In [ ]:
from utils import eval_util_module 
import importlib
importlib.reload(eval_util_module)

In [ ]:
import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.express as px
from urllib.request import urlopen
import json
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)
with urlopen('https://raw.githubusercontent.com/PublicaMundi/MappingAPI/master/data/geojson/us-states.json') as response:
    states = json.load(response)

# remove HI, AK, PR
states['features'] = [
    feat for feat in states['features']
    if feat['id'] not in ['02', '72', '15']
]

In [ ]:
feat = 'sp1_night_rham_noppt'

quantile = np.concatenate([[0.0,0.01,0.03],np.arange(0.05,0.95,0.03),[0.95,0.97,0.99,1.0]])
print(quantile)

warm_state = ['CA','AZ','NM','TX','KS','OK','MO','AR','LA','KY','TN','MS','AL','GA','FL','SC','NC','VA']

param = {'random':[40,41,42,43],
         'control':['lon','lat','month_cos','month_sin','lac_dim']}

control_var = param['control']
feat_var = ['tmin','tmax_ssrd','rh_am','ag_wind_2m']
sub_cols = control_var + feat_var

In [ ]:
from matplotlib.patches import Rectangle
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerBase
import matplotlib.pyplot as plt

# Custom handler to overlay line on top of patch
class HandlerPatchLine(HandlerBase):
    def create_artists(self, legend, orig_handle, x0, y0, width, height, fontsize, trans):
        patch = Rectangle([x0, y0], width, height,
                          facecolor=orig_handle[0], edgecolor='none', lw=0,
                          transform=trans, alpha=0.7)

        line = Line2D([x0, x0 + width -1.2], [y0 + height / 2] * 2,
                      color=orig_handle[1], lw=2, transform=trans)

        return [patch, line]

In [ ]:
heat_color = '#b2182b' #'#d7191c'
cold_color = '#2166ac' #'mediumblue' #'#2c7bb6'
error_heat = 'tomato'
error_cold = 'skyblue'

# 2. Import data

In [ ]:
df = pd.read_parquet('3_output/5_final_herd_detrend_df_full_cow_weight.gzip')

In [ ]:
## opt_condition:
opt_candidates = pd.read_csv('3_output/6_grid_search_output.csv', index_col=0)
warm_cutoff = opt_candidates.loc[opt_candidates['div'] =='warm'].sort_values(by='wpyield', ascending=False).iloc[0,:][feat_var].values
print('warm opt :', warm_cutoff)
cool_cutoff = opt_candidates.loc[opt_candidates['div'] =='cool'].sort_values(by='wpyield', ascending=False).iloc[0,:][feat_var].values
print('cool opt :', cool_cutoff)


# 3. ALE (Figure 2)

In [ ]:
## read ale of main model
ale_df = pd.read_csv('3_output/5_1_weighted_ale.csv')
ale_df['quantile'] = ale_df['quantile'].astype(float).round(3)
ale_df['boot_num'] = 0 # main

In [ ]:
## combining ale_boot and ale_df:
ale_boot = pd.read_parquet('3_output/7_uncertainty_boot_ale.gzip')
ale_boot['quantile'] = ale_boot['quantile'].astype(float).round(3)
lower = ale_boot.groupby(['div','feat_abv','quantile'])['ale'].quantile(0.025).reset_index(name='lower')
higher =ale_boot.groupby(['div','feat_abv','quantile'])['ale'].quantile(0.975).reset_index(name='upper')
ale_df = pd.merge(ale_df, pd.merge(lower, higher, on=['div','feat_abv','quantile']), on=['div','feat_abv','quantile'], how='outer')

In [ ]:


def plot_ale_sensitivity(ale_df, feat, feat_name, div=None, opt_cond=None,
                        ylim=None, xlim=None, output_prefix='ale'):
    """
    Plot ALE sensitivity with rug plot.
    """
    
    # Setup figure
    fig, ax = plt.subplots(figsize=(7, 5), nrows=2, ncols=1,
                           gridspec_kw={'height_ratios': [4, 0.4], 'hspace': 0},
                           sharex=True)
    plt.margins(x=0.02)
    
    for sp in ('bottom', 'top', 'right', 'left'):
        ax[0].spines[sp].set_color('k')
    
    ax[0].axhline(0, color='lightgrey', linestyle='--', linewidth=1)
    
    # === Prepare data ===
    if div is not None:
        plot_df = ale_df.loc[(ale_df['div'] == div) & (ale_df['feat_abv'] == feat)].copy()
        line_color = 'palevioletred' if div == 'warm' else 'skyblue'
    else:
        plot_df = ale_df.loc[ale_df['feat_abv'] == feat].copy()
        line_color = None
        
    plot_df = plot_df.sort_values('values').reset_index(drop=True)
    
    # Convert from log to percent change: %Δ = (exp(Δ_log) - 1) * 100
    cols_to_convert = ['ale']
    if 'lower' in plot_df.columns and 'upper' in plot_df.columns:
        cols_to_convert.extend(['lower', 'upper'])
    
    for col in cols_to_convert:
        plot_df[col + '_pct'] = (np.exp(plot_df[col]) - 1.0) * 100.0
    
    # Main line
    ax[0].plot(plot_df['values'], 
               plot_df['ale_pct'], color='black', linewidth=1.5)
    
    # Add optimum line if provided
    if opt_cond is not None and div is not None:
        opt_val = opt_cond[feat]
        if not np.isnan(opt_val):
            ax[0].axvline(opt_val, color='darkgreen', linestyle='--', 
                         linewidth=2, alpha=0.8)
    
    # Add confidence bands if available
    if 'lower_pct' in plot_df.columns and 'upper_pct' in plot_df.columns and line_color:
        ax[0].fill_between(plot_df['values'], 
                          plot_df['lower_pct'], 
                          plot_df['upper_pct'],
                          color=line_color, alpha=0.5)
    
    # Labels and ticks
    ax[0].set_ylabel('Percent change in milk\nyield per cow per test-day', fontsize=16)
    ax[0].tick_params(axis='both', which='major', labelsize=16, pad=5)
    ax[0].grid(False)
    
    # Set y-limits
    if ylim:
        ax[0].set_ylim(ylim)
    
    # Format y-axis to 1 decimal place
    ax[0].yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    
    # === Rug plot ===
    rug_data = plot_df.sort_values('values', ascending=True)
    rug_data['norm_weight'] = (rug_data['cow_weight'] / rug_data['cow_weight'].sum()).round(3)
    
    height = 0.7
    line_width = 1.06
    for val in rug_data['values']:
        ax[1].vlines(val, 0, height, color='grey', lw=line_width, alpha=0.8)
    
    for sp in ('bottom', 'top', 'right', 'left'):
        ax[1].spines[sp].set_color('k')
    
    ax[1].grid(False)
    ax[1].set_xlabel(feat_name, fontsize=16)
    ax[1].tick_params(axis='x', which='major', labelsize=16, pad=5)
    ax[1].set_yticks([])
    ax[1].set_yticklabels([])
    
    # Set x-limits
    if xlim:
        ax[0].set_xlim(xlim)
    else:
        # Auto-calculate with padding
        min_val = plot_df['values'].min()
        max_val = plot_df['values'].max()
        ax[0].set_xlim(round(min_val) - 1, round(max_val) + 1)
    
    # Add legend ONLY for weather variables (when div is not None)
    if div is not None:
        from matplotlib.lines import Line2D
        from matplotlib.legend_handler import HandlerPatch
        import matplotlib.patches as mpatches
        
        # Create custom legend handles
        cool_patch = mpatches.Patch(facecolor='skyblue', edgecolor='black', 
                                     alpha=0.5, label='Cool Region Response')
        warm_patch = mpatches.Patch(facecolor='palevioletred', edgecolor='black', 
                                     alpha=0.5, label='Warm Region Response')
        opt_line = Line2D([0], [0], color='darkgreen', lw=2, linestyle='--', 
                         label='Optimal Condition')
        
        # Place legend below the rug plot (ax[1])
        fig.legend(handles=[cool_patch, warm_patch, opt_line],
                  loc='lower center',
                  bbox_to_anchor=(0.5, -0.06),
                  frameon=True,
                  framealpha=0.8,
                  edgecolor='dimgrey',
                  fontsize=14,
                  ncol=3)
        
        # Adjust layout to make room for legend
        plt.subplots_adjust(bottom=0.15)
    
    # Save figures
    div_str = f'_{div}' if div else '_national'
    plt.savefig(f'3_data/fig/fig2_{output_prefix}_{div_str}_yield_sensitivity_{feat}_pct.png', 
                dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
# === Control variables ===
control_features = {
    'lac_dim': {
        'name': 'Lactation (Accumulated over Parities)',
        'ylim': (-22, 30),
        'xlim': (0, 912)
    },
    'lat': {
        'name': 'Latitude (degree)',
        'ylim': (-1.5, 1.5),
        'xlim': None
    },
    'lon': {
        'name': 'Longitude (degree)',
        'ylim': (-1.5, 1.5),
        'xlim': None
    }
}

for feat, config in control_features.items():
    plot_ale_sensitivity(
        ale_df=ale_df,
        feat=feat,
        feat_name=config['name'],
        div=None,  # No division for control variables
        ylim=config['ylim'],
        xlim=config['xlim'],
        output_prefix='ale'
    )


In [ ]:
# %%
# === Weather variables ===
weather_features = {
    'tmin': {
        'name': 'Nighttime Temperature (°C)',
        'ylim': (-5.8, 2.5),
        'xlim': None
    },
    'tmax_ssrd': {
        'name': 'Daytime Temperature (°K) × Radiation (kW/m²)',
        'ylim': (-2.5, 2.5),
        'xlim': None
    },
    'rh_am': {
        'name': 'Morning Relative Humidity (%)',
        'ylim': (-2.5, 2.5),
        'xlim': None
    },
    'ag_wind_2m': {
        'name': 'Wind at 2m (m/s)',
        'ylim': (-2.5, 2.5),
        'xlim': (0.75, 4.8)
    }
}

for div in ['warm', 'cool']:
    cutoff = opt_candidates.loc[opt_candidates['div'] ==div].sort_values(by='wpyield', ascending=False).iloc[0,:][feat_var]
    print(cutoff)
    for feat, config in weather_features.items():
        plot_ale_sensitivity(
            ale_df=ale_df,
            feat=feat,
            feat_name=config['name'],
            div=div,
            opt_cond=cutoff,  # Optional: add if you have this
            ylim=config['ylim'],
            xlim=config['xlim'],
            output_prefix='ale'
        )

# 4. Relative yield loss

## 4-1. Calculating predicted yields:

In [ ]:
## model
model = xgb.Booster()
model.load_model(f'3_output/5_xbg_model_full.json')
model.set_param({"device": "cpu"})
model.set_param({'n_jobs':-1})

In [ ]:
target = 'herd_milk_resid'
train_df = xgb.DMatrix(df[sub_cols], df['herd_milk_resid'])
df['p_milk'] = model.predict(train_df)
eval_util_module.evaluate(df[target], df['p_milk'])
del train_df

In [ ]:
## optimum for regional splits:
train_df = df[np.concatenate([['div'], sub_cols])]
train_df.loc[train_df['div'] == 'warm',feat_var] = warm_cutoff
train_df.loc[train_df['div'] =='cool',feat_var] = cool_cutoff
train_df = xgb.DMatrix(train_df[sub_cols], df['herd_milk_resid'])
df['p_opt_milk'] = model.predict(train_df)
del train_df
df.loc[df['p_milk'] > df['p_opt_milk']].shape[0] / df.shape[0]

In [ ]:
## forcing p_opt_milk to p_milk
df.loc[df['p_opt_milk'] < df['p_milk'], 'p_opt_milk'] = df.loc[df['p_opt_milk'] < df['p_milk']]['p_milk']

In [ ]:
## full milk yield predicted values 
df['opt_milk'] = (df['fitted_herd'] + df['p_opt_milk'])
df['pred_milk'] = (df['fitted_herd'] + df['p_milk'])


## 4-2. Timeseries

In [ ]:

from matplotlib.dates import DateFormatter
# Create a monthly period or month-start timestamp
df['year_month'] = df['date'].dt.to_period('M').dt.to_timestamp()

# Aggregate once with the datetime index
mean_df = df.groupby('year_month', as_index=True)[['herd_milk_resid','p_opt_milk','p_milk']].mean()

# Plot
fig, ax = plt.subplots(figsize=(10,4), tight_layout=True)
ax.plot(mean_df.index, mean_df['herd_milk_resid'], marker='o', linestyle='', color='black', markersize=2, label='Observed Yields')
ax.plot(mean_df.index, mean_df['p_opt_milk'], label='Yields under Optimal Weather', color='darkgreen',alpha=0.8)
ax.plot(mean_df.index, mean_df['p_milk'], label='Yields under Historical Weather', linestyle='--', color='tab:grey')

# Format axes
ax.set_ylabel('ln(kg/cow/test-day)', fontsize=15)
ax.set_xlabel('Year–Month',fontsize=15)
ax.tick_params(labelsize=15, which='major',width=2, direction='out', length=8)
ax.legend()
ax.grid(False)

# Optional: better tick formatting
ax.xaxis.set_major_formatter(DateFormatter('%Y-%m'))
# plt.xticks(rotation=45)
ax.set_xmargin(0.01)  
plt.savefig('3_output/fig/SI_timeseries_prediction.png', dpi=90, bbox_inches='tight')
plt.show()


## 4-3. Calculating relative yield loss

In [ ]:
df['rel_loss_percent'] = np.nan
df.loc[:,'rel_loss_percent'] = ((np.exp(df['p_opt_milk']) - np.exp(df['p_milk'])) / np.exp(df['p_opt_milk'])) * 100

In [ ]:
## defining heat and cold stress
stress = 'stress'
df.loc[:,'stress'] = np.nan

df.loc[(df['div'] == 'warm') & (df['tmin'].round(5) > np.round(warm_cutoff[0],5)) ,stress]='Heat'
df.loc[(df['div'] == 'warm') & (df['tmin'].round(5) < np.round(warm_cutoff[0],5)),stress] = 'Cold'

df.loc[(df['div'] == 'cool') & (df['tmin'].round(5) > np.round(cool_cutoff[0],5)) ,stress]='Heat'
df.loc[(df['div'] == 'cool') & (df['tmin'].round(5) < np.round(cool_cutoff[0],5)) ,stress] ='Cold'
# df.loc[df['stress'].isnull()][['GEOID','date']+sub_cols]

# 5. Quantile based relative yield loss (Figure 3)

In [ ]:
quantile_y = np.concatenate([np.arange(0.00,0.98,0.02),[0.98,0.99,0.999]])
stress = 'stress'
yloss = 'rel_loss_percent'

## 5-1. Main model output

In [ ]:
bin_edge_df = pd.DataFrame()
stress_df = pd.DataFrame()
for region in ['warm','cool']:
    for stress_div in ['Cold','Heat']:
        quantile_values = np.quantile(df.loc[(df[stress] == stress_div) & (df['div'] == region)][yloss], q=quantile_y)
        quantile_values = quantile_values*-1

        # Accumulated data points corresponding to each percentile
        N = len(df.loc[(df[stress] == stress_div) & (df['div'] == region)][yloss])

        # Compute bin widths in data points
        bin_sizes = np.diff([0] + quantile_y) * N
        bin_sizes = np.round(bin_sizes).astype(int)

        # Get cumulative bin edges:
        bin_edges = np.concatenate([[0], np.cumsum(bin_sizes)])
        print(bin_edges)

        # For mirrored butterfly:
        if stress_div == 'Cold':
            x_pos = -bin_edges[::-1] / 1e6  # Cold goes left, so reverse
            quantile_values = quantile_values[::-1]
            quantile_labels = [f"{int(q*100) if q<0.998 else '99.99'}%" for q in quantile_y][::-1]
            bin_edges = bin_edges[::-1]
        else:
            x_pos = bin_edges /1e6
            quantile_labels = [f"{int(q*100) if q<0.998 else '99.99'}%" for q in quantile_y]

        data_points1 = np.round(bin_edges /1e6,1)


        stress_df = pd.concat([stress_df, pd.DataFrame(data={'div':region, 'stress':stress_div, 'x_pos':x_pos,
                                                             'labels':quantile_labels,
                                                             'value':quantile_values,
                                                             'n':bin_edges,
                                                             'data_points':data_points1})], ignore_index=True)

In [ ]:
stress_df.to_parquet('3_output/8_main_quantile_stress_df.gzip', compression='gzip')

In [ ]:
## cow level
df.groupby(['stress'])[yloss].mean().reset_index()

## 5-2. Bootstrap output 

In [ ]:
boot_stress_df = pd.read_parquet('3_output/7_uncertainty_boot_quantile_cow_test_day_loss.gzip')

In [ ]:
lower = boot_stress_df.groupby(['div','stress','labels'])['value'].quantile(0.025).reset_index(name='lower')
higher = boot_stress_df.groupby(['div','stress','labels'])['value'].quantile(0.975).reset_index(name='upper')

## 5-3. Plot

In [ ]:
## merging
stress_df = pd.merge(stress_df, pd.merge(lower, higher, on=['div',stress,'labels'], how='outer'),
         on=['div',stress,'labels'], how='outer')

In [ ]:
stress_df

In [ ]:
## formatting - drop "%" in labels 
stress_df['labels'] = stress_df['labels'].str[:-1]
stress_df['lab'] = stress_df['labels']
## putting negative sign for cold stress:
stress_df.loc[stress_df['stress'] == 'Cold', 'lab'] = '-' + stress_df.loc[stress_df['stress'] == 'Cold', 'lab']
stress_df['lab'] = stress_df['lab'].astype(float)

In [ ]:
## flagging significance:
stress_df.loc[(stress_df['lower'].isin([0,-0])) | (stress_df['upper'].isin([0, -0])), 'sig'] = 'No'
stress_df.loc[stress_df['sig'].isnull(),'sig'] = 'Yes'

In [ ]:
sig_cutoff = {'cool_Heat':[], 'cool_Cold':[], 'warm_Heat':[], 'warm_Cold':[]}
for region in ['cool','warm']:
    fig, ax = plt.subplots(figsize=(8, 5))
    xticks = []
    xticklabels = []
    for spine in ax.spines.values():
        spine.set_linestyle('-')
        spine.set_linewidth(2)
        spine.set_color('black')

    for stress_div, color in zip(['Cold', 'Heat'], [cold_color, heat_color]):
        if stress_div == 'Cold':
            error_color = error_cold
        else:
            error_color = error_heat

        sig_cut = stress_df.loc[(stress_df['div'] == region) & (stress_df['stress'] == stress_div) & (stress_df['sig'] == 'No')]['labels'].astype(float).sort_values().iloc[-1]
        sig_cutoff[region+'_'+stress_div] = stress_df[(stress_df['stress'] == stress_div) & (stress_df['div'] == region)  & (stress_df['labels'].astype(float) == sig_cut)]['value'].iloc[0]
        
        print(sig_cut)
        
        # Significant data (above cutoff)
        sig_data = stress_df[
            (stress_df['stress'] == stress_div) & 
            (stress_df['div'] == region) & 
            (stress_df['labels'].astype(float) >= sig_cut)
        ].sort_values('x_pos').copy()  
        
        # Non-significant data (below cutoff)
        nonsig_data = stress_df[
            (stress_df['stress'] == stress_div) & 
            (stress_df['div'] == region) & 
            (stress_df['labels'].astype(float) <= sig_cut)
        ].sort_values('x_pos').copy()  
        
        # Error band for significant
        if len(sig_data) > 0:
            ax.fill_between(sig_data['x_pos'],
                           sig_data['lower'], 
                           sig_data['upper'], 
                           color=error_color, alpha=0.7)
        
        # Error band for non-significant
        if len(nonsig_data) > 0:
            ax.fill_between(nonsig_data['x_pos'],
                           nonsig_data['lower'], 
                           nonsig_data['upper'], 
                           color='grey', alpha=0.5)
        
        # Main line - significant
        if len(sig_data) > 0:
            ax.plot(sig_data['x_pos'],
                   sig_data['value'], 
                   color=color, lw=2, zorder=3, label=stress_div)
        
        # Main line - non-significant
        if len(nonsig_data) > 0:
            ax.plot(nonsig_data['x_pos'],
                   nonsig_data['value'], 
                   color='grey', lw=2, ls='--', zorder=2)

    ax.margins(x=0.02)
    ax.set_xticks(stress_df.loc[(stress_df['div'] == region) & (stress_df['labels'].isin(['99.99','50','0']))].drop_duplicates(subset=['x_pos'])['x_pos'])
    ax.set_xticklabels(stress_df.loc[(stress_df['div'] == region) & (stress_df['labels'].isin(['99.99','50','0']))].drop_duplicates(subset=['x_pos'])['labels'])
    ax.tick_params(which='both', labelsize=14, pad=3, length=8, color='black', width=2)
    ax.set_xlabel('Percentile (%)', fontsize=15)
    ax.set_ylabel('Relative yield loss\nper cow per test-day (%)', fontsize=15, labelpad=3)
    ax.grid(False)
    ax.set_ylim(-22,0.5)
     
    # Define Cold and Heat handles (box color, line color)
    cold_handle = (error_cold, cold_color)
    heat_handle = (error_heat, heat_color)

    # Add the combined legend
    if region == 'warm':
        ax.legend(
            handles=[cold_handle, heat_handle],
            labels=['Cold', 'Heat'],
            loc='lower left',
            frameon=True,
            handler_map={tuple: HandlerPatchLine()},
            framealpha=0.8,
            edgecolor='dimgrey',
            fontsize=15,
            ncols=2
        )

    # Secondary axis for number of data points
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(stress_df.loc[(stress_df['div'] == region) & (stress_df['labels'].isin(['99.99','50','0']))].drop_duplicates(subset=['x_pos'])['x_pos'])
    ax2.set_xticklabels(stress_df.loc[(stress_df['div'] == region) & (stress_df['labels'].isin(['99.99','50','0']))].drop_duplicates(subset=['x_pos'])['data_points'])
    ax2.set_xlabel('Accumulated Number of Data Points (Million)', fontsize=15, labelpad=8)
    ax2.grid(False)

    # Position secondary axis below main axis
    ax2.spines['bottom'].set_position(('outward', 60))
    ax2.xaxis.set_ticks_position('bottom')
    ax2.xaxis.set_label_position('bottom')
    ax2.spines['bottom'].set_color('black')
    ax2.spines['bottom'].set_linewidth(2)
    ax2.tick_params(axis='x', which='major', pad=8, length=8, width=2, direction='in', labelsize=14)

    plt.tight_layout()
    # plt.savefig(f'3_output/fig/fig3_percentile_{region}.png',dpi=300,bbox_inches='tight')
    
    plt.show()

# 6. Temporal and spatial relative yield loss


## 6-1. Yearly relative yield loss

In [ ]:
yearly_df = (df.loc[df['year'] < 2024].groupby(['state_abv','GEOID','id','year',stress])[yloss].mean()
             .reset_index().groupby(['year',stress])[yloss].mean().reset_index())
yearly_df[yloss] *= -1


In [ ]:
boot_yearly_df = pd.read_parquet('3_output/7_uncertainty_boot_yearly_loss.gzip')

In [ ]:
var = yloss

n_year = 2023-2000+1
spacing = 1.8  # Try 1.5 or 2 for more space

x = np.arange(n_year) * spacing
width = 0.7

# Example data for two groups
final_loss_cold = yearly_df.loc[(yearly_df[stress] == 'Cold')][['year',var]]
final_loss_heat = yearly_df.loc[(yearly_df[stress] == 'Heat')][['year',var]]

lower_cold = boot_yearly_df.loc[boot_yearly_df[stress] == 'Cold'].groupby(['year'])[var].quantile(0.025).reset_index()
upper_cold = boot_yearly_df.loc[boot_yearly_df[stress] == 'Cold'].groupby(['year'])[var].quantile(0.975).reset_index()
lower_heat = boot_yearly_df.loc[boot_yearly_df[stress] == 'Heat'].groupby(['year'])[var].quantile(0.025).reset_index()
upper_heat = boot_yearly_df.loc[boot_yearly_df[stress] == 'Heat'].groupby(['year'])[var].quantile(0.975).reset_index()
# print(lower_cold, upper_cold, lower_heat, upper_heat)

fig, ax = plt.subplots(figsize=(12, 4), tight_layout=True)
bar1 = ax.bar(x - width/2, final_loss_cold[var].values, width=width, color=cold_color, label='Cold')
bar2 = ax.bar(x + width/2, final_loss_heat[var].values, width=width, color=heat_color, label='Heat')

# vlines for uncertainties (as above)
for xi, y, low, high in zip(x - width/2, final_loss_cold[var].values, lower_cold[var].values, upper_cold[var].values):
    ax.vlines(xi, low, high, color=error_cold, lw=1.3, alpha=0.7)
    ax.scatter([xi], [low], color=error_cold, marker='_', s=35, lw=1.3, alpha=0.7)
    ax.scatter([xi], [high], color=error_cold, marker='_', s=35, lw=1.3, alpha=0.7)
for xi, y, low, high in zip(x + width/2, final_loss_heat[var].values, lower_heat[var].values, upper_heat[var].values):
    ax.vlines(xi, low, high, color=error_heat, lw=1.3, alpha=0.7)
    ax.scatter([xi], [low], color=error_heat, marker='_', s=35, lw=1.3, alpha=0.7)
    ax.scatter([xi], [high], color=error_heat, marker='_', s=35, lw=1.3, alpha=0.7)

# Set custom x-ticks at centers of grouped bars
ax.set_xticks(x)
ax.set_xlim(x[0] - spacing*0.6, x[-1] + spacing*0.6)
ax.set_xticklabels([str(i) for i in range(2000, 2000+n_year)], fontsize=15, rotation=45)

ax.set_xlabel('Year', fontsize=15)
ax.set_ylim(-4,0)
ax.set_ylabel('Average relative yield loss\nper cow per test-day (%)', fontsize=15, labelpad=5)
ax.legend(loc='lower right', 
          frameon=True, framealpha=0.8,#bbox_to_anchor=(0.02, 0.02),
            edgecolor='dimgrey',fontsize=12, ncols=2)
ax.grid(False)
ax.tick_params(which='both', labelsize=15, pad=3, length=5)
plt.tight_layout()
plt.savefig(f'3_output/fig/SI_yearly_relative_yield_loss.png',dpi=90,bbox_inches='tight')
plt.show()

In [ ]:
del yearly_df, boot_yearly_df
gc.collect()

## 6-2. Monthly relative yield loss (Figure 4)

In [ ]:
## average like this since there could be multiple test-day records in a given month:
month_df = (df.loc[(df['year'] < 2024)].groupby(['state_abv','GEOID','id','year','month',stress])[yloss].mean()
            .reset_index().groupby(['year','month',stress])[yloss].mean()
            .reset_index().groupby(['month',stress])[yloss].mean().reset_index())
           
month_df[yloss] *= -1

In [ ]:
test = df.loc[(df['year'] < 2024)].groupby(['state_abv','GEOID','id','year','month',stress])[yloss].mean().reset_index()
test.groupby(['stress','month']).size()

In [ ]:
full_index = pd.MultiIndex.from_product([np.arange(1,13,1), ['Cold','Heat']], names=['month', 'stress'])
month_df = month_df.set_index(['month','stress']).reindex(full_index, fill_value=0).reset_index()

In [ ]:
data_count = df.loc[(df[stress].isin(['Heat','Cold']))].groupby([stress,'month'])[yloss].count().reset_index()
full_index = pd.MultiIndex.from_product([np.arange(1,13,1), ['Cold','Heat']], names=['month', 'stress'])
data_count = data_count.set_index(['month','stress']).reindex(full_index, fill_value=0).reset_index()

In [ ]:
boot_month_df = pd.read_parquet('3_output/7_uncertainty_boot_month_loss.gzip')

In [ ]:
boot_month_df.loc[boot_month_df['stress'].isnull(),'stress'] =boot_month_df.loc[boot_month_df['stress'].isnull()]['stress']
full_index = pd.MultiIndex.from_product([boot_month_df['boot_num'].unique(),np.arange(1,13,1), ['Cold','Heat']], names=['boot_num','month', 'stress'])
boot_month_df = boot_month_df.set_index(['boot_num','month','stress']).reindex(full_index, fill_value=0).reset_index()

In [ ]:
import calendar  # for month names

# Get month labels
month_labels = [calendar.month_abbr[i] for i in range(1, 13)]

# Setup
var = yloss
n_months = 12
spacing = 1.6
x = np.arange(n_months) * spacing
width = 0.7

# Example data for two groups
final_loss_cold = month_df.loc[month_df[stress] == 'Cold'][['month',var]]
final_loss_heat = month_df.loc[month_df[stress] == 'Heat'][['month',var]]

lower_cold = boot_month_df.loc[boot_month_df[stress] == 'Cold'].groupby(['month'])[var].quantile(0.025).reset_index()
upper_cold = boot_month_df.loc[boot_month_df[stress] == 'Cold'].groupby(['month'])[var].quantile(0.975).reset_index()
lower_heat = boot_month_df.loc[boot_month_df[stress] == 'Heat'].groupby(['month'])[var].quantile(0.025).reset_index()
upper_heat = boot_month_df.loc[boot_month_df[stress] == 'Heat'].groupby(['month'])[var].quantile(0.975).reset_index()



# Prepare figure with two vertically stacked subplots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8.8, 6), sharex=True, gridspec_kw={'height_ratios': [4.2, 0.8], 'hspace':0.01})

### --- Top Plot: Bar plot with error bars --- ###
bar1 = ax1.bar(x - width/2, final_loss_cold[var].values, width=width, color=cold_color, label='Cold')
bar2 = ax1.bar(x + width/2, final_loss_heat[var].values, width=width, color=heat_color, label='Heat')

# vlines for uncertainties (as above)
for xi, y, low, high in zip(x - width/2, final_loss_cold[var].values, lower_cold[var].values, upper_cold[var].values):
    ax1.vlines(xi, low, high, color=error_cold, lw=1.5, alpha=0.7)
    ax1.scatter([xi], [low], color=error_cold, marker='_', s=35, lw=1.8, alpha=0.7)
    ax1.scatter([xi], [high], color=error_cold, marker='_', s=35, lw=1.8, alpha=0.7)
for xi, y, low, high in zip(x + width/2, final_loss_heat[var].values, lower_heat[var].values, upper_heat[var].values):
    ax1.vlines(xi, low, high, color=error_heat, lw=1.5, alpha=0.7)
    ax1.scatter([xi], [low], color=error_heat, marker='_', s=35, lw=1.8, alpha=0.7)
    ax1.scatter([xi], [high], color=error_heat, marker='_', s=35, lw=1.8, alpha=0.7)


ax1.set_ylabel('Average relative yield loss\nper cow per test-day (%)', fontsize=18, color='black',labelpad=10)

ax1.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, 1.2),   # center horizontally, move above
    frameon=True,
    framealpha=0.8,
    edgecolor='dimgrey',
    fontsize=16,
    ncols=2
)
ax1.grid(False)
ax1.tick_params(which='both', labelsize=16, pad=3, length=8, color='black', width=2)
ax1.tick_params(axis='x', which='both', bottom=False, labelbottom=False)
# ax1.spines['bottom'].set_visible(False) # hide??
#ax1.spines['top'].set_linestyle('--')
ax1.spines['top'].set_color('dimgrey')
ax1.spines['top'].set_linewidth(1.0) #
ax1.spines['bottom'].set_visible(False)
ax1.spines['left'].set_color('dimgrey')
ax1.spines['left'].set_linewidth(1.0)
ax1.spines['right'].set_color('dimgrey')
ax1.spines['right'].set_linewidth(1.0)

### --- Bottom Plot: Exposure fraction --- ###

for stress_div in ['Heat','Cold']:
    g = data_count.loc[data_count[stress] == stress_div].sort_values('month')
    color = heat_color if stress_div == 'Heat' else cold_color
    y = g[yloss].values / 1e6
    
    # x_smooth = np.linspace(1, 12, 100)
    x_smooth = np.linspace(x[0] -width, x[-1] +width, 200)
    x_months_align = (np.arange(1,13,1) -1) * spacing
    x_months_align = np.concatenate([x_months_align, [x_smooth[-1]]])
    y_ext = np.concatenate([y, [y[0]]])
    pchip = PchipInterpolator(x_months_align, y_ext)
    # pchip = PchipInterpolator(x_months, y)
    # y_smooth = pchip(x_smooth)
    y_smooth = pchip(x_smooth)
    # y_smooth[-1] = y_smooth[0]
    
    ax2.plot(x_smooth , y_smooth, label=stress_div, color=color, linewidth=0)
    ax2.fill_between(x_smooth, y_smooth, color=color, alpha=0.5, linewidth=0)
    

ax2.set_ylabel('Million  ', fontsize=14, labelpad=5, color='black')
ax2.set_xlabel('Month', fontsize=15, labelpad=7)
ax2.set_xticks(x)
ax2.set_xticklabels(month_labels, fontsize=14, color='black')
ax2.grid(False)
ax2.tick_params(which='both', labelsize=14, pad=3, length=8, width =2, color='black')
ax2.yaxis.tick_right()
ax2.yaxis.set_label_position('right')
ax2.spines['top'].set_visible(False)
ax2.spines['left'].set_color('dimgrey')
ax2.spines['left'].set_linewidth(1.0)
ax2.spines['right'].set_color('dimgrey')
ax2.spines['right'].set_linewidth(1.0)
ax2.spines['bottom'].set_color('dimgrey')
ax2.spines['bottom'].set_linewidth(1.0)

ax1.set_xlim(x[0] - width -0.3 , x[-1]+width+0.3)
ax2.set_xlim(x[0] - width -0.3, x[-1]+width+0.3)
fig.suptitle("Distribution of relative yield loss", fontsize=14, y=0.28,x=0.33)
plt.tight_layout(h_pad=0.1)
plt.savefig(f'3_output/fig/fig4_monthly_relative_yield_loss.png',dpi=300,bbox_inches='tight')

plt.show()

In [ ]:
del month_df, boot_month_df
gc.collect()

## 6-3. County-level relative yield loss (Figure 4)

In [ ]:
## county yearly average map
county_map = (df.loc[df['year'] < 2024].groupby(['state_abv','GEOID','id','year',stress])[yloss].mean()
              .reset_index().groupby(['state_abv','GEOID','year',stress])[yloss].mean()
              .reset_index().groupby(['state_abv','GEOID',stress])[yloss].mean().reset_index()
             )
county_map[yloss] *= -1

In [ ]:
county_map['div'] = np.nan
county_map.loc[county_map['state_abv'].isin(warm_state), 'div'] = 'warm'
county_map.loc[~county_map['state_abv'].isin(warm_state),'div'] = 'cool'
county_map.groupby(['div','stress'])[yloss].mean()

In [ ]:
full_scale = px.colors.cyclical.IceFire

# Get the "Fire" part (red/yellow half)
fire_only = full_scale[len(full_scale)//2:][1:][::-1]

In [ ]:
div_maxmin = {'Heat':[fire_only,-4.5],
              'Cold':['ice',-4]}

for stress in ['Heat','Cold']:
    plot_df = county_map.loc[(county_map['stress'] == stress) ].copy()
    
    county = pd.read_parquet('1_data/full_state_county_names.gzip')
    county = (county.loc[(~county['GEOID'].isnull()) & (~county['State_abv'].isin(['HI','GU','MP','PR','AK']))]
              .reset_index(drop=True)[['GEOID','County_name']].drop_duplicates())
    plot_df = plot_df.merge(county, on=['GEOID'], how='outer')
    plot_df.loc[plot_df[yloss].isnull(), yloss] =-999
    
    
    fig = go.Figure(go.Choropleth( locationmode='geojson-id', geojson=counties,
                                  locations=plot_df.loc[plot_df[yloss] != -999]['GEOID'],
                                  z=plot_df.loc[plot_df[yloss] != -999][yloss], 
                                  colorscale=div_maxmin[stress][0],
                                  zmin=div_maxmin[stress][1], zmax=0,
                                  colorbar={'outlinecolor':'black', 'outlinewidth':2,'tickfont':dict(size=20,family='Arial'),
                                           'orientation':'h','xanchor':'center','y':-0.45,'x':0.46,'len':0.6,'thickness':20},
                                  marker_line_width=0.2,
                                  colorbar_title=dict(text='Average relative yield loss<br>per cow per test-day (%)', font_family='Arial', side='top',
                                                      font_size=15)
                                 ))

    fig.add_trace(go.Choropleth( locationmode='geojson-id', geojson=counties,
                                  locations=plot_df.loc[plot_df[yloss] == -999]['GEOID'],
                                  z=plot_df.loc[plot_df[yloss] == -999][yloss],
                               colorscale = [[0, 'rgb(239,239,239)'], [1, 'rgb(239,239,239)']] , showscale=False, marker_line_width=0.2))

    for feature in states['features']:
        geom_type = feature['geometry']['type']
        coords = feature['geometry']['coordinates']

        if geom_type == 'Polygon':
            # Single polygon: list of rings
            rings = coords
        elif geom_type == 'MultiPolygon':
            # Multiple polygons: list of list of rings
            rings = [ring for polygon in coords for ring in polygon]
        else:
            continue  # Skip if not a polygon

        for ring in rings:
            try:
                lons, lats = zip(*ring)
                fig.add_trace(go.Scattergeo(
                    lon=list(lons),
                    lat=list(lats),
                    mode='lines',
                    line=dict(color='black', width=0.6),
                    showlegend=False
                ))
            except Exception:
                continue  # In case of bad geometry

    # 4. Finalize layout
    fig.update_geos(
        visible=False,
        scope='usa',
        showsubunits=False,  
        resolution=110
    )

    fig.update_layout(margin=dict(l=0,r=0,t=0,b=0), paper_bgcolor="white")
    fig.write_image(f'3_output/fig/fig4_county_map_{stress}.png', scale=3)
    fig.show()

In [ ]:
del county_map

# 7. Economic damage


## 7-1) State-level annaul production loss
- revenue is proportional to production.

In [ ]:
cow_weight = df[['GEOID','id','year','final_weight']].drop_duplicates()
cow_weight['state_id'] = cow_weight['GEOID'].str[:2]
print(cow_weight.shape)

In [ ]:
## changing names:
df['log_opt_milk'] = df['fitted_herd'] + df['p_opt_milk']
df['log_pred_milk'] = df['fitted_herd'] + df['p_milk']
df['loss'] = np.nan
df['opt_milk'] = np.exp(df['log_opt_milk'])
df['pred_milk'] = np.exp(df['log_pred_milk'])
df['loss'] = df['opt_milk'] - df['pred_milk']
stress='stress'

In [ ]:
## sum loss per cow-year-stress
yearly_loss = (df.groupby(['state_abv','GEOID','id','year',stress])['loss'].sum()
                 .reset_index()
                )
## merging weights
yearly_loss = pd.merge(yearly_loss, cow_weight, on=['GEOID','id','year'],
                         how='left')
print('any null ?', yearly_loss.loc[yearly_loss['final_weight'].isnull()].shape[0])
yearly_loss['Q_loss'] = yearly_loss['loss'] * yearly_loss['final_weight']

## state-year-stress production loss
state_loss = yearly_loss.groupby(['state_abv','state_id','year','stress'])['Q_loss'].sum().reset_index()

In [ ]:
## calculating optimal production prediction
yearly_opt = (df.groupby(['state_abv','GEOID','id','year'])['opt_milk'].sum()
                 .reset_index()
                )
yearly_opt = pd.merge(yearly_opt, cow_weight, on=['GEOID','id','year'],
                         how='left')
yearly_opt['Q_opt'] = yearly_opt['opt_milk'] * yearly_opt['final_weight']
yearly_opt = yearly_opt.groupby(['state_abv','state_id','year'])['Q_opt'].sum().reset_index()

In [ ]:
## merging:
state_prod = pd.merge(state_loss, yearly_opt, on=['state_abv','state_id','year'], how='left')

state_prod['frac_kts'] = state_prod['Q_loss'] / state_prod['Q_opt'] 

In [ ]:
del yearly_loss
gc.collect()

## 7-2. USDA milk sales revenue

In [ ]:
sales = pd.read_csv('1_data/usda_nass_survey_milk_sales_dollars_state_level_2000_2024_annual_level_downloaded_8Aug2025.csv', index_col=0)
sales = sales[['Year','Geo Level','State ANSI','Data Item','Domain','Value']].copy().rename(columns={'Year':'year','Value':'sales','State ANSI':'state_id'})
print(sales['Geo Level'].unique(), sales['Data Item'].unique(), sales['Domain'].unique())
sales = sales.loc[~sales['state_id'].isnull()].copy().reset_index(drop=True)
sales['state_id'] = sales['state_id'].astype(int).astype(str).str.zfill(2)
print('remove D :', sales.loc[sales['sales'] == ' (D)'].shape[0])
sales = sales[sales['sales'] != ' (D)'].copy()
sales['sales'] = sales['sales'].replace(",","", regex=True).astype('float64')

In [ ]:
## calculating r_opt:
sales_df = pd.merge(sales[['state_id','year','sales']], 
                    state_prod[['state_abv','state_id','year','frac_kts']].groupby(['state_abv','state_id','year'])['frac_kts'].sum().reset_index(),
                    on=['state_id','year'], how='right')

## 7-3. Producer Price Index
- Monthly level => average it into annual level

In [ ]:
ppi = pd.read_csv('1_data/producer_price_index_for_raw_milk_not_seasonally_adjusted_WPU016101.csv', index_col=0)
ppi = ppi.rename(columns={'Year':'year','Value':'ppi'})
ppi = ppi.groupby(['year'])['ppi'].mean().reset_index()
ppi['adj_ppi'] = ppi.loc[ppi['year'] == 2023]['ppi'].values / ppi['ppi']

In [ ]:
## merging with sales_df
sales_df = pd.merge(sales_df, ppi, on=['year'], how='left')
sales_df['adj_sales'] = (sales_df['sales'] * sales_df['adj_ppi']).astype(float).astype(int)
sales_df['adj_sales_opt'] = (sales_df['adj_sales'] / (1 - sales_df['frac_kts'])).astype(float).astype(int)

In [ ]:
state_prod = state_prod.merge(sales_df.drop(columns=['frac_kts']), on=['state_abv','state_id','year'], how='left')
state_prod['sales_loss'] = (state_prod['adj_sales_opt'] * state_prod['frac_kts']).astype(float).astype(int)
state_prod['temp_loss'] = state_prod['adj_sales_opt'] - state_prod['adj_sales']
state_prod['temp_loss_sum'] = state_prod.groupby(['state_abv','year'])['sales_loss'].transform('sum')

In [ ]:
state_prod.to_parquet('3_output/8_econ_loss.gzip',compression='gzip')

## 7-4. Bootstrap data

In [ ]:
boot_state_df = pd.read_parquet('3_output/7_uncertainty_boot_econ_loss.gzip')

## 7-5. Economic loss 

In [ ]:
## changing name
state_df = state_prod.copy()
del state_prod

### 7-5-1. Annual loss

In [ ]:
plot_df = state_df.loc[state_df['year'] < 2024].groupby(['year','stress'])['sales_loss'].sum().rename('econ_loss').reset_index()
plot_df['econ_loss'] = plot_df['econ_loss'] / 1e6

boot_plot_df = (boot_state_df.loc[(boot_state_df['year'] < 2024)]
                .groupby(['boot_num','year','stress'])['sales_loss'].sum()
                .rename('econ_loss').reset_index()
               )
boot_plot_df['econ_loss'] = boot_plot_df['econ_loss'] / 1e6

In [ ]:
## annaul average:
print('main model :', plot_df.groupby(['year'])['econ_loss'].sum().mean())
boot_plot_df.groupby(['boot_num','year'])['econ_loss'].sum().reset_index().groupby(['boot_num'])['econ_loss'].mean().quantile([0.025, 0.975])

In [ ]:
print('early years of main :', plot_df.loc[plot_df['year'].isin([2000,2001,2002])].groupby(['year'])['econ_loss'].sum().mean().round(3),
      'later years of main :', plot_df.loc[plot_df['year'].isin([2021,2022,2023])].groupby(['year'])['econ_loss'].sum().mean().round(3),
      'early years of boot :', boot_plot_df.loc[(boot_plot_df['year'].isin([2000,2001,2002]))].groupby(['boot_num','year'])['econ_loss'].sum().reset_index().groupby(['boot_num'])['econ_loss'].mean().quantile(q=[0.025,0.975]).round(3),
      'later years of boot :', boot_plot_df.loc[(boot_plot_df['year'].isin([2021,2022,20023]))].groupby(['boot_num','year'])['econ_loss'].sum().reset_index().groupby(['boot_num'])['econ_loss'].mean().quantile(q=[0.025,0.975]).round(3)
     )
      

In [ ]:
temp = state_df[['state_abv','year','adj_sales_opt','temp_loss_sum']].drop_duplicates().groupby(['year'])[['adj_sales_opt','temp_loss_sum']].sum().reset_index()
temp['frac'] = temp['temp_loss_sum'] / temp['adj_sales_opt']
print('share of national revenue :',
      temp.loc[temp['year'].isin([2000,2001,2002])]['frac'].mean().round(3) * 100,
      temp.loc[temp['year'].isin([2021,2022,2023])]['frac'].mean().round(3) * 100,
     )
print('share of national revenue :',
      temp.loc[temp['year'].isin([2000,2001,2002])]['adj_sales_opt'].mean() / 1e9,
      temp.loc[temp['year'].isin([2021,2022,2023])]['adj_sales_opt'].mean() / 1e9,
     )


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sel_years = sorted(plot_df['year'].unique())
n_years = len(sel_years)

# Custom bar spacing
bar_spacing = 1.5
positions = np.arange(n_years) * bar_spacing  # wider spacing
bar_width = 0.8  # optional

#positions = np.arange(n_years)
xticks = positions
xticklabels = [str(int(y)) for y in sel_years]

# Initialize bottoms for stacking (per year)
bottoms = np.zeros(n_years)

for stress_div, color, error_color in zip(['Cold', 'Heat'], [cold_color, heat_color], [error_cold, error_heat]):

    # === Compute total loss per year ===
    total_loss = [
        plot_df.loc[(plot_df['stress'] == stress_div) & (plot_df['year'] == yr), 'econ_loss'].sum()
        for yr in sel_years
    ]

    # Plot bar
    bar = ax.bar(
        positions,
        total_loss,
        bottom=bottoms,
        color=color,
        label=stress_div
    )

    # Plot one error bar per year
    for i_yr, yr in enumerate(sel_years):
        # Filter bootstrap rows for this stress and year
        boots = boot_plot_df.loc[
            (boot_plot_df['stress'] == stress_div) & (boot_plot_df['year'] == yr)
        ]

        lower, upper = np.percentile(boots['econ_loss'], [0.25, 97.5])

        # Plot error line at stacked position
        base = bottoms[i_yr]
        ax.plot([positions[i_yr], positions[i_yr]], [base + lower, base + upper],
                color=error_color, lw=1.5, alpha=0.7)
        ax.plot([positions[i_yr]-0.1, positions[i_yr]+0.1], [base + lower, base + lower],
                color=error_color, lw=1, alpha=0.7)
        ax.plot([positions[i_yr]-0.1, positions[i_yr]+0.1], [base + upper, base + upper],
                color=error_color, lw=1, alpha=0.7)

#     # Update bottom for stacking
    bottoms += np.array(total_loss)

# === Axis formatting ===
ax.set_xticks(xticks)
ax.set_xlim(positions[0] - bar_width, positions[-1] + bar_width)
ax.set_xticklabels(xticklabels, rotation=45)
ax.tick_params(which='both', labelsize=15, pad=3, length=5)
# ax.set_title('Economic Damage of US Milk Production Loss', fontsize=15, pad=10)
ax.set_xlabel('Year', fontsize=15, labelpad=5)
ax.set_ylabel('Economic Damages\n(billion 2023$/yr)', fontsize=15, labelpad=5)
ax.grid(False)
legend_elems = [
    Patch(facecolor=heat_color, label='Heat'),
    Patch(facecolor=cold_color, label='Cold')
]
ax.legend(handles=legend_elems, loc='upper left', edgecolor='dimgrey', framealpha=0.8, fontsize=14)
ax.spines['top'].set_color('dimgrey')
ax.spines['top'].set_linewidth(1.0) #
ax.spines['bottom'].set_color('dimgrey')
ax.spines['bottom'].set_linewidth(1.0) #
ax.spines['left'].set_color('dimgrey')
ax.spines['left'].set_linewidth(1.0)
ax.spines['right'].set_color('dimgrey')
ax.spines['right'].set_linewidth(1.0)
plt.tight_layout()
plt.savefig(f'3_output/fig/fig5_yearly_economic_damage.png',dpi=300,bbox_inches='tight')
plt.show()

### 7-5-2. State-average loss

In [ ]:
## state-average milk yield loss:
plot_df = (state_df.loc[(state_df['year'] < 2024)].groupby(['state_abv','stress'])['frac_kts'].mean().rename('econ_frac').reset_index())

plot_df['econ_frac']  = plot_df['econ_frac'] * 100
plot_df.groupby(['stress'])['econ_frac'].describe()

In [ ]:
## naming
county = pd.read_parquet('1_data/full_state_county_names.gzip')
county = county.loc[(~county['GEOID'].isnull()) & (~county['State_abv'].isin(['HI','GU','MP','PR','AK']))].reset_index(drop=True)[['State_abv','State_name']].drop_duplicates().rename(columns={'State_abv':'state_abv'})
county.loc[county['state_abv'] == 'DC', 'State_name'] = 'District of Columbia'
plot_df = plot_df.merge(county, on=['state_abv'], how='left')

In [ ]:
## counting counties:
county_num = df[['state_abv','GEOID','stress']].drop_duplicates().groupby(['state_abv','stress'])['GEOID'].nunique().reset_index()

In [ ]:
plot_df = plot_df.merge(county_num, on=['state_abv','stress'], how='left')
plot_df['small_sample']= 'No'
plot_df.loc[plot_df['GEOID'] < 5, 'small_sample'] = 'Yes'
plot_df.loc[plot_df['small_sample'] == 'Yes']

In [ ]:
full_scale = px.colors.cyclical.IceFire

# Get the "Fire" part (red/yellow half)
fire_only = full_scale[len(full_scale)//2:][1:]
ice_only = px.colors.sequential.ice_r

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, to_hex

def rgb_to_hex(rgb_str):
    # "rgb(8,48,107)" → "#08306b"
    nums = rgb_str.strip("rgb()").split(",")
    r,g,b = [int(n) for n in nums]
    return to_hex([r/255, g/255, b/255])


ice_only = [rgb_to_hex(c) for c in ice_only]
cmap_fire = LinearSegmentedColormap.from_list("fire_only", fire_only)
cmap_ice = LinearSegmentedColormap.from_list("ice_only", ice_only)

In [ ]:
import geopandas as gpd
import matplotlib as mpl

stress_div = 'Heat'
var = 'econ_frac'
target_states = plot_df.loc[(plot_df['stress'] == stress_div) & (plot_df['small_sample'] == 'Yes')]['State_name'].unique() # hatch these
print(target_states)
if stress_div == 'Heat':
    tick_label = [0.1, 0.2, 0.3, 0.4,0.5]
    vmax = 0.6
else:
    tick_label = [0.1, 0.2, 0.3, 0.4,0.5]
    vmax = 0.6
    
map_color = cmap_fire if stress_div == 'Heat' else cmap_ice

# 1) Read states GeoJSON into GeoDataFrame
# If you have a dict (like your `states`) -> use gpd.GeoDataFrame.from_features
if isinstance(states, dict):
    g_states = gpd.GeoDataFrame.from_features(states['features'], crs="EPSG:4326").rename(columns={'name':'State_name'})
else:
    g_states = gpd.read_file(states)  # path to file

# 2) Merge your data
g = g_states.merge(plot_df.loc[plot_df['stress'] == stress_div][['State_name', var]], 
                   on='State_name', how='left')

# 3) Plot choropleth (missing counties as lightgrey via missing_kwds)
fig, ax = plt.subplots(figsize=(10, 6))
g.plot(
    column=var,
    ax=ax,
    cmap=map_color,         
    legend=False,
    edgecolor='black',
    linewidth=0.5,
    vmin = 0,
    vmax = vmax,
    missing_kwds={'color':'gainsboro', 'edgecolor':'black'}
)

if len(target_states) >0:
    # 4) Hatch selected states (overlay facecolor=none + hatch pattern)
    g[g['State_name'].isin(target_states)].plot(
        ax=ax,
        facecolor='none',
        edgecolor='dimgrey',
        linewidth=0.5,
        hatch='///'
    )

sm = plt.cm.ScalarMappable(
    cmap=map_color,
    norm=mpl.colors.Normalize(vmin=0, vmax=tick_label[-1])
)
cbar = fig.colorbar(
    sm, 
    ax=ax, 
    orientation="horizontal",
    fraction=0.06,   # shrink relative size
    pad=0.05,         # distance from map
    ticks= tick_label
)
cbar.set_label("Economic Damages per State (% of state revenue 2023$)", fontsize=14, labelpad=5)
cbar.ax.xaxis.set_label_position('top')
cbar.outline.set_edgecolor('black')
cbar.ax.tick_params(labelsize=13, pad=5)


# 5) Cosmetics
ax.set_axis_off()
fig.tight_layout()
plt.savefig(f'3_output/fig/fig5_state_map_economic_damage_share_{stress_div}.png',dpi=300,bbox_inches='tight')
plt.show()

### 7-5-3. Top 10 states

In [ ]:
plot_df = state_df.loc[state_df['year'] < 2024].groupby(['state_abv','stress'])['sales_loss'].mean().rename('econ_loss').reset_index()
plot_df['econ_loss'] = plot_df['econ_loss'] / 1e6

In [ ]:
plot_df = plot_df.merge(county_num, on=['state_abv','stress'], how='left')
plot_df['small_sample']= 'No'
plot_df.loc[plot_df['GEOID'] < 5, 'small_sample'] = 'Yes'
plot_df.loc[plot_df['small_sample'] == 'Yes'].head()

In [ ]:
boot_plot_df = boot_state_df.loc[(boot_state_df['year'] < 2024)].groupby(['boot_num','state_abv','stress'])['sales_loss'].mean().rename('econ_loss').reset_index()
boot_plot_df['econ_loss'] =boot_plot_df['econ_loss']/1e6

In [ ]:
boot_summary = (
    boot_plot_df
    .groupby(['state_abv', 'stress'])['econ_loss']
    .agg([
        ('econ_loss_p5', lambda x: np.percentile(x, 2.5)),
        ('econ_loss_p95', lambda x: np.percentile(x, 97.5))
    ])
    .reset_index()
)

In [ ]:
from matplotlib.patches import Patch

stress_div = 'Heat'
bar_width = 0.7

color = heat_color if stress_div == 'Heat' else cold_color
error_color = error_heat if stress_div == 'Heat' else error_cold

top10 = (plot_df.loc[plot_df['stress'] == stress_div]
         .nlargest(10, 'econ_loss')
         .sort_values('econ_loss', ascending=True)
         .reset_index(drop=True))

index = np.arange(len(top10))

plt.figure(figsize=(4, 5.2))
bars = plt.barh(index, top10['econ_loss'], bar_width,
                color=color, linewidth=0,
                alpha=0.85, label=stress_div)

# ---- HATCH selected states ----
hatch_states = set(plot_df.loc[
    (plot_df['stress'] == stress_div) & (plot_df['small_sample'] == 'Yes'),
    'state_abv'
].unique())

for i, abv in enumerate(top10['state_abv']):
    if abv in hatch_states:
        bars[i].set_linewidth(0.01)
        bars[i].set_facecolor(color)
        bars[i].set_hatch('///')

# ---- Error bars (your manual whiskers) ----
for i, row in top10.iterrows():
    lo = float(boot_summary.loc[
        (boot_summary['state_abv'] == row['state_abv']) &
        (boot_summary['stress'] == stress_div), 'econ_loss_p5'
    ])
    hi = float(boot_summary.loc[
        (boot_summary['state_abv'] == row['state_abv']) &
        (boot_summary['stress'] == stress_div), 'econ_loss_p95'
    ])
    plt.plot([lo, hi], [index[i], index[i]], color=error_color, lw=2, alpha=0.8, zorder=3)
    plt.plot([lo, lo], [index[i]-0.10, index[i]+0.10], color=error_color, lw=2.5, alpha=0.8, zorder=3)
    plt.plot([hi, hi], [index[i]-0.10, index[i]+0.10], color=error_color, lw=2.5, alpha=0.8, zorder=3)

# ---- Axis & labels ----
plt.yticks(index, top10['state_abv'], fontsize=15)
plt.xlabel('Economic Damages\n(million 2023$/yr)', fontsize=15)
plt.tick_params(which='both', labelsize=14, pad=3, length=5)
plt.grid(False)

# ---- Spine customization ----
ax = plt.gca()
for side in ['top', 'bottom', 'left', 'right']:
    ax.spines[side].set_color('dimgrey')
    ax.spines[side].set_linewidth(1.0)
ax.set_ylim(-0.5, len(top10) - 0.5)
plt.tight_layout()
plt.savefig(f'3_output/fig/fig5_top_10_economic_damage_{stress_div}.png', dpi=300, bbox_inches='tight')
plt.show()

